In [1]:
!pip install -q -U transformers accelerate requests

import os
import json
import glob
import time
import torch
import torch.nn.functional as F
from PIL import Image
from transformers import AutoProcessor, LlavaForConditionalGeneration

device = "cuda" if torch.cuda.is_available() else "cpu"
print("device:", device)

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.3/12.3 MB 110.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 394.3/394.3 kB 29.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.1/73.1 kB 5.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 516.0/516.0 kB 34.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.4/3.4 MB 62.7 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
bigframes 2.39.0 requires google-cloud-bigquery-storage<3.0.0,>=2.30.0, which is not installed.
google-adk 1.29.0 requires google-cloud-bigquery-storage>=2.0.0, which is not installed.
google-colab 1.0.0 requires jupyter-server==2.14.0, but you have jupyter-server 2.12.5 which is incompatible.
google-colab 1.0.0 requires pandas==2.2.2, but you have pandas 2.3.3 which is incompatible.
google-colab 1.0.0 requires requests==2.3

In [2]:
MODEL_NAME = "llava-hf/llava-1.5-7b-hf"

processor = AutoProcessor.from_pretrained(MODEL_NAME)
model = LlavaForConditionalGeneration.from_pretrained(
    MODEL_NAME,
    torch_dtype=torch.float16,
    device_map="auto",
    max_memory={0: "12GiB", 1: "12GiB"},
)
model.eval()
for p in model.parameters():
    p.requires_grad_(False)

ln_f = model.model.language_model.norm
lm_head = model.lm_head

n_layer = model.config.text_config.num_hidden_layers
hidden_dim = model.config.text_config.hidden_size
print(n_layer, "language model layers,", hidden_dim, "hidden dim")

processor_config.json:   0%|          | 0.00/173 [00:00<?, ?B/s]

chat_template.json:   0%|          | 0.00/701 [00:00<?, ?B/s]

chat_template.jinja:   0%|          | 0.00/674 [00:00<?, ?B/s]

preprocessor_config.json:   0%|          | 0.00/505 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/950 [00:00<?, ?B/s]

...

model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 3 files:   0%|          | 0/3 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/686 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/141 [00:00<?, ?B/s]

32 language model layers, 4096 hidden dim


In [3]:
_encoder_cache = {}

def _vision_tower_hook(module, inputs, output):
    _encoder_cache["patch_embeddings"] = output.last_hidden_state[:, 1:, :].detach().cpu()

_hook_handle = model.model.vision_tower.register_forward_hook(_vision_tower_hook)
print("hook registered on model.vision_tower")

hook registered on model.vision_tower


In [4]:
img_dir = "/kaggle/input/datasets/nadaibrahim/coco2014/val2014/val2014"


def load_image(path):
    return Image.open(path).convert("RGB")


def build_inputs(image, text):
    conversation = [{"role": "user", "content": [{"type": "image"}, {"type": "text", "text": text}]}]
    prompt_text = processor.apply_chat_template(conversation, add_generation_prompt=True)
    return processor(text=prompt_text, images=image, return_tensors="pt").to(model.device, torch.float16)

In [5]:
HALLUC_TYPE = "object"
USERNAME = "nocturnalnerd18"

if HALLUC_TYPE == "relation":
    !wget https://cs.stanford.edu/people/rak248/VG_100K/images.zip -O /kaggle/working/VG_100K.zip
    !wget https://cs.stanford.edu/people/rak248/VG_100K_2/images2.zip -O /kaggle/working/VG_100K_2.zip
    !unzip -q -o /kaggle/working/VG_100K.zip -d /kaggle/temp/
    !unzip -q -o /kaggle/working/VG_100K_2.zip -d /kaggle/temp/
    for zpath in ["/kaggle/working/VG_100K.zip", "/kaggle/working/VG_100K_2.zip"]:
        if os.path.exists(zpath):
            os.remove(zpath)
    print("Visual Genome downloaded, extracted, zips removed")
else:
    print(f"HALLUC_TYPE={HALLUC_TYPE!r} -- Visual Genome not needed, skipping download")

HALLUC_TYPE='object' -- Visual Genome not needed, skipping download


In [6]:
AMBER_IMAGE_DIR = "/kaggle/input/datasets/nocturnalnerd18/amber-hallucination/image"
VG_IMAGE_DIRS = ["/kaggle/temp", "/kaggle/temp/VG_100K_2"]

def find_vg_image_path(image_id: str) -> str:
    for d in VG_IMAGE_DIRS:
        candidate = os.path.join(d, f"{image_id}.jpg")
        if os.path.exists(candidate):
            return candidate
    return None


def resolve_image_path(q: dict) -> str:
    qid = q["question_id"]
    if qid.startswith("pope_"):
        return os.path.join(img_dir, f"{q['image_id']}.jpg")
    elif qid.startswith("amber_"):
        return os.path.join(AMBER_IMAGE_DIR, f"{q['image_id']}.jpg")
    elif qid.startswith("reefknot_"):
        return find_vg_image_path(q["image_id"])
    return None

In [7]:
QUESTIONS_PATH = "/kaggle/input/datasets/nocturnalnerd18/vlm-questions/all_questions.json"

with open(QUESTIONS_PATH) as f:
    all_questions = json.load(f)

assert len(all_questions) == 17492, "attach vlm-questions version 2"
questions_subset = [q for q in all_questions if q["hallucination_type"] == HALLUC_TYPE and q["answer_format"] == "yes_no"]
print(len(all_questions), "total questions loaded,", len(questions_subset), f"{HALLUC_TYPE} yes/no questions selected")

17492 total questions loaded, 3000 object yes/no questions selected


In [8]:
EMPTY_IDS = build_inputs(Image.new("RGB", (336, 336)), text="")["input_ids"][0].tolist()


def question_span(ids):
    n = min(len(ids), len(EMPTY_IDS))
    start = next(i for i in range(n) if ids[i] != EMPTY_IDS[i])
    tail = next(i for i in range(n) if ids[-1 - i] != EMPTY_IDS[-1 - i])
    return start, len(ids) - tail


@torch.no_grad()
def cache_question_aware(image_path, image_id, question_id, question_text):
    image = load_image(image_path)
    inputs = build_inputs(image, text=question_text)
    ids = inputs["input_ids"][0].tolist()
    q_start, q_end = question_span(ids)

    _encoder_cache.clear()
    outputs = model(
        input_ids=inputs["input_ids"], pixel_values=inputs["pixel_values"],
        attention_mask=inputs["attention_mask"], output_hidden_states=True,
    )
    last_logits = outputs.logits[0, -1]

    image_token_id = model.config.image_token_index
    image_positions = (inputs["input_ids"][0] == image_token_id).nonzero(as_tuple=True)[0]

    p1_hidden = torch.stack([layer[0, -1, :].half().cpu() for layer in outputs.hidden_states])
    p2_hidden = torch.stack([layer[0, q_start:q_end, :].mean(dim=0).half().cpu() for layer in outputs.hidden_states])
    pooled_image_hidden = torch.stack([layer[0, image_positions, :].mean(dim=0).half().cpu() for layer in outputs.hidden_states])
    pooled_encoder_embedding = _encoder_cache["patch_embeddings"].mean(dim=1).half()

    return {
        "image_id": image_id, "question_id": question_id, "question_text": question_text,
        "answer_text": processor.tokenizer.decode(last_logits.argmax().item()).strip(),
        "span_text": processor.tokenizer.decode(ids[q_start:q_end]),
        "p1_hidden": p1_hidden,
        "p2_hidden": p2_hidden,
        "pooled_image_hidden": pooled_image_hidden,
        "pooled_encoder_embedding": pooled_encoder_embedding,
        "answer_logits": last_logits.half().cpu(),
    }

In [9]:
OLD_CACHE_DIR = "/kaggle/input/datasets/nocturnalnerd18/discriminative-cache-amber"
question_by_id = {q["question_id"]: q for q in all_questions}

old_entries = torch.load(sorted(glob.glob(os.path.join(OLD_CACHE_DIR, "*.pt")))[0])[::4]

t0 = time.time()
new_entries = []
for e in old_entries:
    q = question_by_id[e["question_id"]]
    new_entries.append(cache_question_aware(resolve_image_path(q), q["image_id"], q["question_id"], q["question_text"]))
sec_per_question = (time.time() - t0) / len(new_entries)

answers_match = sum(e["answer_text"].lower().startswith(n["answer_text"].lower()) for n, e in zip(new_entries, old_entries))
min_cosine = min(F.cosine_similarity(n["pooled_image_hidden"].float(), e["pooled_hidden_states"].float(), dim=-1).min().item() for n, e in zip(new_entries, old_entries))
logit_err = max((lm_head(n["p1_hidden"][-1].to(lm_head.weight.device)).float().cpu() - n["answer_logits"].float()).abs().max().item() for n in new_entries)
layer0_identical = all(torch.equal(n["p1_hidden"][0], new_entries[0]["p1_hidden"][0]) for n in new_entries)
spans_match = sum(n["span_text"].strip() == question_by_id[n["question_id"]]["question_text"].strip() for n in new_entries)

print(f"{len(new_entries)} questions, {sec_per_question:.2f} s/question")
print(f"answers match old cache: {answers_match}/{len(new_entries)}")
print(f"min cosine vs old pooled image states: {min_cosine:.5f}")
print(f"layer-32 p1 through lm_head vs answer_logits, max abs err: {logit_err:.4f}")
print(f"layer-0 p1 identical across questions: {layer0_identical}")
print(f"question span decodes to the question: {spans_match}/{len(new_entries)}")

50 questions, 0.49 s/question
answers match old cache: 50/50
min cosine vs old pooled image states: 1.00000
layer-32 p1 through lm_head vs answer_logits, max abs err: 0.0156
layer-0 p1 identical across questions: True
question span decodes to the question: 50/50


In [10]:
QA_CACHE_DIR = f"/kaggle/working/cache/question_aware_{HALLUC_TYPE}"
os.makedirs(QA_CACHE_DIR, exist_ok=True)

BATCH_SAVE_SIZE = 200
batch_entries, batch_index, skipped, span_mismatch = [], 0, 0, 0
t_start = time.time()

for i, q in enumerate(questions_subset):
    image_path = resolve_image_path(q)
    if image_path is None or not os.path.exists(image_path):
        skipped += 1
        continue

    entry = cache_question_aware(image_path, q["image_id"], q["question_id"], q["question_text"])
    entry.update({k: q[k] for k in ("ground_truth_answer", "answer_format", "hallucination_type")})
    span_mismatch += entry["span_text"].strip() != q["question_text"].strip()
    batch_entries.append(entry)

    if len(batch_entries) >= BATCH_SAVE_SIZE:
        torch.save(batch_entries, os.path.join(QA_CACHE_DIR, f"{HALLUC_TYPE}_batch_{batch_index}.pt"))
        batch_entries, batch_index = [], batch_index + 1

    if i % 100 == 0 and i > 0:
        elapsed = time.time() - t_start
        avg = elapsed / (i + 1)
        remaining = avg * (len(questions_subset) - i - 1)
        print(f"[{i+1}/{len(questions_subset)}] {q['question_id']}: {entry['answer_text']!r} "
              f"({avg:.2f}s/question avg, ETA {remaining/60:.1f} min, skipped: {skipped}, span mismatches: {span_mismatch})")

if batch_entries:
    torch.save(batch_entries, os.path.join(QA_CACHE_DIR, f"{HALLUC_TYPE}_batch_{batch_index}.pt"))

print(f"done -- {batch_index + 1} files for {HALLUC_TYPE}, {skipped} skipped, "
      f"{span_mismatch} span mismatches, {(time.time()-t_start)/60:.1f} min total")

[101/3000] pope_100: 'Yes' (0.50s/question avg, ETA 24.1 min, skipped: 0, span mismatches: 0)
[201/3000] pope_200: 'Yes' (0.52s/question avg, ETA 24.3 min, skipped: 0, span mismatches: 0)
[301/3000] pope_300: 'Yes' (0.53s/question avg, ETA 23.7 min, skipped: 0, span mismatches: 0)
[401/3000] pope_400: 'Yes' (0.53s/question avg, ETA 23.1 min, skipped: 0, span mismatches: 0)
[501/3000] pope_500: 'Yes' (0.54s/question avg, ETA 22.3 min, skipped: 0, span mismatches: 0)
[601/3000] pope_600: 'Yes' (0.54s/question avg, ETA 21.5 min, skipped: 0, span mismatches: 0)
[701/3000] pope_700: 'Yes' (0.54s/question avg, ETA 20.7 min, skipped: 0, span mismatches: 0)
[801/3000] pope_800: 'No' (0.54s/question avg, ETA 19.8 min, skipped: 0, span mismatches: 0)
[901/3000] pope_900: 'Yes' (0.54s/question avg, ETA 18.9 min, skipped: 0, span mismatches: 0)
[1001/3000] pope_1000: 'Yes' (0.54s/question avg, ETA 18.1 min, skipped: 0, span mismatches: 0)
[1101/3000] pope_1100: 'Yes' (0.54s/question avg, ETA 17.2 

In [11]:
assert skipped == 0, f"{skipped} questions skipped, not publishing"
dataset_slug = f"question-aware-cache-{HALLUC_TYPE}"

with open(os.path.join(QA_CACHE_DIR, "dataset-metadata.json"), "w") as f:
    json.dump({
        "title": dataset_slug,
        "id": f"{USERNAME}/{dataset_slug}",
        "licenses": [{"name": "CC0-1.0"}]
    }, f)

!kaggle datasets create -p {QA_CACHE_DIR}

Starting upload for file object_batch_1.pt
100%|█████████████████████████████████████████| 168M/168M [00:01<00:00, 101MB/s]
Upload successful: object_batch_1.pt (168MB)
Starting upload for file object_batch_6.pt
100%|████████████████████████████████████████| 168M/168M [00:02<00:00, 58.9MB/s]
Upload successful: object_batch_6.pt (168MB)
Starting upload for file object_batch_8.pt
100%|█████████████████████████████████████████| 168M/168M [00:01<00:00, 107MB/s]
Upload successful: object_batch_8.pt (168MB)
Starting upload for file object_batch_5.pt
100%|████████████████████████████████████████| 168M/168M [00:02<00:00, 73.6MB/s]
Upload successful: object_batch_5.pt (168MB)
Starting upload for file object_batch_2.pt
100%|████████████████████████████████████████| 168M/168M [00:01<00:00, 88.2MB/s]
Upload successful: object_batch_2.pt (168MB)
Starting upload for file object_batch_4.pt
100%|████████████████████████████████████████| 168M/168M [00:01<00:00, 91.4MB/s]
Upload successful: object_batch